# 제로 에너지 드링크 HPLC 분석

카페인 · 소듐벤조에이트를 표준물 첨가법으로 정량합니다.

**메뉴에서 `런타임 → 모두 실행` 을 누르면 끝까지 돌아갑니다.**
따로 내려받거나 업로드할 파일은 없습니다. 코드와 실측 데이터가
이 노트북 안에 들어 있습니다.

---

### ⚠ 모의 데이터에 관하여

맨 아래 선택 항목에 **모의 데이터 생성기**가 있습니다. 코드가 제대로
도는지 확인하는 용도입니다. 거기서 나온 숫자를 보고서에 실측값으로
적으면 **데이터 조작**입니다. 모의 데이터로 돌린 결과물에는 `SIMULATED`
표시가 자동으로 박히니 지우지 마세요.


## 1단계 — 준비물 설치

분석 라이브러리와 한글 폰트를 깝니다. 1~2분 걸립니다.

폰트를 까는 이유는 matplotlib이 한글 폰트 없이 그래프를 그리면 글자가
네모(□□□)로 나오기 때문입니다.


In [ ]:
!pip install -q numpy pandas matplotlib scipy
!apt-get install -y -qq fonts-nanum > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1

import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
print('한글 폰트:', 'NanumGothic' in {f.name for f in fm.fontManager.ttflist})
print('설치 완료')


## 2단계 — 코드 풀어놓기

분석 코드와 실측 데이터를 `/content/hplc_project` 에 풉니다.
인터넷에서 받아오지 않고 이 노트북 안에 심어 둔 것을 씁니다.

푼 뒤에는 왼쪽 폴더 아이콘에서 파일을 직접 열어 고칠 수 있습니다.


In [ ]:
import base64, io, zipfile, os, sys, shutil

PROJECT_ZIP_B64 = (
    "..."
)

PROJECT = '/content/hplc_project'
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ZIP_B64))) as zf:
    zf.extractall(PROJECT)
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

for root, _, names in os.walk(PROJECT):
    for n in sorted(names):
        print(os.path.relpath(os.path.join(root, n), PROJECT))


## 3단계 — 실험 설계 점검

머무름 시간, 머무름 계수 k', 첨가 농도 적정성, 표준액 조제량을 출력합니다.
파일은 만들지 않고 화면에만 나옵니다.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

!python -m hplc design-check


## 4단계 — 실측 데이터 분석

실험에서 나온 피크 면적 96행(음료 3종 × 성분 2종 × 첨가 4수준 × 주입 4회)을
분석합니다. 표준물 첨가법으로 x절편을 역산해 실제 농도를 구합니다.

> 증류수 바탕 검량선 데이터가 아직 없어서 **변환 상수 칸은 비어 나옵니다.**
> 농도 역산은 검량선을 쓰지 않으므로 결과 자체는 그대로 유효합니다.
> 검량선을 구하셨으면 6단계 아래의 안내를 보세요.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

!python -m hplc analyze data/measured_peak_areas.csv


## 5단계 — 그림 보기

표준물 첨가법 그래프와 잔차 그래프를 노트북 안에서 봅니다.
그래프를 우클릭하면 이미지로 저장할 수 있습니다.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

from IPython.display import Image, display, Markdown
import glob

figs = sorted(glob.glob('output/*.png'))
if not figs:
    print('그림이 없습니다. 4단계를 먼저 실행하세요.')
for f in figs:
    display(Markdown(f'### {os.path.basename(f)}'))
    display(Image(filename=f))


## 6단계 — 보고서 읽기

보고서 7장에 그대로 옮길 수 있는 표들입니다. 자동 점검 결과도 맨 아래
붙어 있으니 꼭 확인하세요.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

from IPython.display import Markdown, display
import glob

reports = sorted(glob.glob('output/*report.md'))
if reports:
    display(Markdown(open(reports[0], encoding='utf-8').read()))
else:
    print('보고서가 없습니다. 4단계를 먼저 실행하세요.')


## 7단계 — 결과 내려받기

보고서·표·그림을 ZIP 하나로 묶어 저장합니다.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

import shutil
from google.colab import files

if not os.path.isdir('output'):
    print('결과가 없습니다. 4단계를 먼저 실행하세요.')
else:
    shutil.make_archive('hplc_results', 'zip', 'output')
    files.download('hplc_results.zip')


---

# 여기부터는 선택 항목

`런타임 → 모두 실행` 은 위 7단계까지로 충분합니다. 아래는 필요할 때만
개별로 실행하세요.

## 값 고치기 — 라벨 표시량, 검량선, 실험 조건

왼쪽 **폴더 아이콘 → hplc → config.py** 를 더블클릭하면 편집기가 열립니다.
고친 뒤 4단계부터 다시 실행하면 반영됩니다.

| 고칠 곳 | 무엇 |
|---|---|
| `DRINKS` | 캔의 실제 카페인 표기량(mg)과 용량(mL) ← **지금 자리표시자입니다** |
| `EXTERNAL_CALIBRATION` | 검량선 계수 `{"caffeine": (기울기, y절편), ...}` |
| `HPLCConditions` | 이동상 비율, 유량, 오븐 온도, 검출 파장 |
| `expected_rt_min` | 실측 머무름 시간 |
| `PrepConditions` | 희석배수, 첨가 농도 |

> Colab에서 고친 내용은 런타임이 끊기면 사라집니다. 계속 쓸 값이면
> GitHub의 `config.py` 도 같이 고쳐 두세요.


## (선택) 내 CSV 올려서 분석하기

검량선 행을 추가했거나 재측정한 데이터가 있으면 여기에 올리세요.
실행하면 파일 선택 버튼이 나옵니다.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

import shutil
from google.colab import files

shutil.rmtree('output', ignore_errors=True)
print('CSV 파일을 선택하세요.')
uploaded = files.upload()
csv_name = next(n for n in uploaded if n.lower().endswith('.csv'))

# Colab 은 같은 이름이 있으면 'xxx (1).csv' 로 저장하므로,
# 올린 내용을 항상 같은 이름으로 직접 써 준다.
MEASURED = 'uploaded.csv'
with open(MEASURED, 'wb') as fh:
    fh.write(uploaded[csv_name])
print(f'{csv_name} -> {MEASURED} ({len(uploaded[csv_name]):,} bytes)')

!python -m hplc analyze uploaded.csv


## (선택) 빈 입력표 받기

새로 실험할 때 쓸 빈 CSV입니다. 구글 스프레드시트로 열어
`peak_area` 열을 채우세요. 피크가 없으면 `0` 을 적습니다.


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

from google.colab import files

!python -m hplc template
files.download('data/peak_areas_template.csv')


## (선택) 모의 데이터로 코드 검증

정답을 아는 가짜 데이터를 넣고 그 정답이 되돌아오는지 확인합니다.
**여기서 나오는 숫자는 실측값이 아닙니다.**


In [ ]:
import os
os.chdir('/content/hplc_project')   # 셀 순서가 어긋나도 되도록 매번 이동

!python -m pytest tests/ -q
!python -m hplc demo
